# 14 · Missing values and encoding: testing combinations with a for loop
Teacher's advice (*Modelling and Evaluation 2*, "For loop on preprocessing"): grid search only tunes
hyperparameters, but **how missing values are treated and how categories are encoded** also changes the result.
This notebook loops over combinations and scores each one on **1-notch accuracy** (the DSC metric) on the validation set.

**Current treatment (what notebook 02 does now):**
1. **Domain rules in `clean()`:** structural missing values → 0 (pool, garden, apartment surface);
   impossible values → missing (Price ≤ 0, 0 facades); indicator `no_3d_data`; patrimony code dropped.
2. **Pipeline:** numeric → **median** (from the training set); text → **most frequent** category; then one-hot.
3. **HistGradientBoosting:** numeric NaN kept (the model learns where missing values should go); text NaN → own code.

**Weakness found:** the flood columns are text. "Most frequent" fills the 39% missing (all of Wallonia and Brussels)
with the most common value ("no risk"), so the model is told those buildings have no flood risk. A separate
**"missing" category** keeps the information that the value is unknown.

**Techniques tested here:**
| Group | Options |
|---|---|
| Numeric imputation | median, mean, **KNN imputer** (values of similar buildings), **iterative imputer** (predicts each missing value from the other columns), none (HGB handles NaN itself) |
| Missing indicator | with / without a 0/1 column "this value was missing" |
| Text imputation | most frequent, constant "missing" category |
| Encoding | one-hot, ordinal, target encoding |

Every choice is fitted on the **training sample only** and judged on the validation sample, with two models:
logistic regression (sensitive to preprocessing) and HistGradientBoosting (the likely final model).

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *
import itertools, time, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")

d = load_prepared()
data = d["data"]
y_train, y_val = d["y_train"], d["y_val"]
X_tr, X_va = data.iloc[d["indices_train"]], data.iloc[d["indices_val"]]
X_tr_s, y_tr_s = stratified_sample(X_tr, y_train, 30000)       # samples keep each run to about a minute
X_va_s, y_va_s = stratified_sample(X_va, y_val, 10000)

cat_cols = data.select_dtypes(exclude="number").columns.tolist()
num_cols = data.select_dtypes(include="number").columns.tolist()

: 

## 1 · Where values are still missing after `clean()`

In [ ]:
miss = X_tr.isna().mean()
miss = miss[miss > 0].sort_values(ascending=False)
overview = pd.DataFrame({
    "missing_%": (miss * 100).round(1),
    "type": ["numeric" if c in num_cols else "text" for c in miss.index],
    "current treatment": ["median" if c in num_cols else "most frequent category" for c in miss.index],
})
print(f"{len(miss)} columns still have missing values ({(overview['type'] == 'numeric').sum()} numeric, "
      f"{(overview['type'] == 'text').sum()} text)")
overview

In [ ]:
# Is missingness itself informative? Share of E/F/G when a value is missing vs. present
info = {}
for c in miss.index[:15]:
    m = X_tr[c].isna().to_numpy()
    info[c] = {"EFG if missing": np.isin(y_train[m], POOR).mean(), "EFG if present": np.isin(y_train[~m], POOR).mean()}
pd.DataFrame(info).T.round(3)

A large gap between the two columns means *that* a value is missing already says something about the label.
That information is lost by plain imputation and kept by a missing indicator, a "missing" category, or HGB's native NaN handling.

## 2 · The experiment function

In [ ]:
def run(num_impute="median", add_indicator=False, cat_impute="most_frequent", encoding="onehot"):
    t = time.time()
    pre = build_preprocessor(num_cols, cat_cols, num_impute, add_indicator, cat_impute, encoding)
    xt = pre.fit_transform(X_tr_s, y_tr_s)                   # y only used by target encoding
    xv = pre.transform(X_va_s)
    row = {"num_impute": num_impute, "indicator": add_indicator, "cat_impute": cat_impute,
           "encoding": encoding, "n_features": xt.shape[1]}
    if num_impute != "none":                                 # logistic regression cannot handle NaN
        lr = LogisticRegression(C=0.1, max_iter=500).fit(xt, y_tr_s)
        row["LR_1notch"] = one_notch_accuracy(y_va_s, predict_one_notch(class_scores(lr, xv)))
    xt_d = xt.toarray() if hasattr(xt, "toarray") else xt
    xv_d = xv.toarray() if hasattr(xv, "toarray") else xv
    hgb = HistGradientBoostingClassifier(max_iter=300, early_stopping=True, random_state=0).fit(xt_d, y_tr_s)
    proba = class_scores(hgb, xv_d)
    row["HGB_1notch"] = one_notch_accuracy(y_va_s, predict_one_notch(proba))
    row["HGB_macroF1"] = macro_f1(y_va_s, proba.idxmax(axis=1))
    row["seconds"] = round(time.time() - t)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})
    return row

results = []

## 3 · For loop, stage 1: numeric imputation × missing indicator
Text treatment fixed at the current setup (most frequent + one-hot).

In [ ]:
for num_impute, indicator in itertools.product(["median", "mean", "knn", "iterative"], [False, True]):
    results.append(run(num_impute, indicator))
results.append(run("none"))                                  # HGB learns from NaN itself
stage1 = pd.DataFrame(results)
stage1.sort_values("HGB_1notch", ascending=False).round(4)

## 4 · For loop, stage 2: text imputation × encoding
Uses the best numeric setting from stage 1 (by HGB 1-notch accuracy).

In [ ]:
best_num = stage1.sort_values("HGB_1notch", ascending=False).iloc[0]
print("Best numeric setting:", best_num["num_impute"], "| indicator:", best_num["indicator"])

for cat_impute, encoding in itertools.product(["most_frequent", "constant"], ["onehot", "ordinal", "target"]):
    results.append(run(best_num["num_impute"], bool(best_num["indicator"]), cat_impute, encoding))
all_results = pd.DataFrame(results).drop_duplicates(["num_impute", "indicator", "cat_impute", "encoding"], keep="last")
all_results.sort_values("HGB_1notch", ascending=False).round(4)

## 5 · Optional: the full product of all options
Every combination (5 × 2 × 2 × 3 = 60 runs, roughly an hour). Set `FULL = True` only if time allows;
the staged search above usually finds the same winner.

In [ ]:
FULL = False
if FULL:
    for combo in itertools.product(["median", "mean", "knn", "iterative", "none"], [False, True],
                                   ["most_frequent", "constant"], ["onehot", "ordinal", "target"]):
        if combo[0] == "none" and combo[1]:
            continue
        results.append(run(*combo))
    all_results = pd.DataFrame(results).drop_duplicates(["num_impute", "indicator", "cat_impute", "encoding"], keep="last")

## 6 · Compare and choose

In [ ]:
all_results["setup"] = (all_results["num_impute"] + np.where(all_results["indicator"], " + indicator", "")
                        + " | " + all_results["cat_impute"] + " | " + all_results["encoding"])
plot_df = all_results.set_index("setup").sort_values("HGB_1notch")
plot_df[["HGB_1notch", "LR_1notch"]].plot(kind="barh", figsize=(9, 0.4 * len(plot_df) + 1))
current = plot_df.loc["median | most_frequent | onehot", "HGB_1notch"]
plt.axvline(current, color="grey", linestyle="--", label="current setup (HGB)")
lo = np.nanmin(plot_df[["HGB_1notch", "LR_1notch"]].to_numpy())
plt.xlim(lo - 0.01, min(1, plot_df["HGB_1notch"].max() + 0.01))
plt.xlabel("validation 1-notch accuracy"); plt.title("Missing-value and encoding combinations"); plt.legend()
plt.tight_layout(); plt.show()

best = all_results.sort_values("HGB_1notch", ascending=False).iloc[0]
print(f"Current setup: {current:.4f} | best: {best['HGB_1notch']:.4f} ({best['HGB_1notch'] - current:+.4f})")
print("Best combination:", dict(best[["num_impute", "indicator", "cat_impute", "encoding"]]))

**How to read it:** differences below about 0.003 are noise on a 10k validation sample. Prefer the simpler setup when
two are equally good (median over iterative, one-hot over target encoding).

## 7 · Use the winner in notebook 02
Replace the `preprocessor = ColumnTransformer(...)` block in Step 4 of notebook 02 with the line printed below,
and change `preprocessor.fit_transform(data.iloc[indices_train])` to
`preprocessor.fit_transform(data.iloc[indices_train], status.iloc[indices_train])` (needed for target encoding,
harmless otherwise). Then rerun 02 and the model notebooks.
If the best numeric setting is `none`, only HistGradientBoosting can use it: keep `median` in 02 for the other models.

In [ ]:
num_choice = best["num_impute"] if best["num_impute"] != "none" else "median"
print(f'preprocessor = build_preprocessor(num_cols, cat_cols, num_impute="{num_choice}", '
      f'add_indicator={bool(best["indicator"])}, cat_impute="{best["cat_impute"]}", encoding="{best["encoding"]}")')